# PSP8 — Análise: respostas às perguntas de negócio P1–P4

**Projeto:** Sistema de Suporte à Decisão para Avaliação de Viabilidade de Nova Unidade de Infraestrutura de Dados · UnB / PSP8.

Este notebook lê a **camada prata persistida no Google Drive** (gerada e verificada pelo notebook `PSP8_Pipeline_Bronze_Prata_Colab.ipynb`) e responde às quatro perguntas aprovadas:

| ID | Pergunta |
|---|---|
| P1 | Qual é o comportamento histórico dos custos, riscos e prazos dos projetos? |
| P2 | Como um cenário de nova unidade se posiciona em relação a esse histórico? |
| P3 | Qual é a exposição geral do cenário e quais revisões são recomendadas? |
| P4 | Como alterações de custo, risco e prazo mudam a classificação e as recomendações? |

**Leia antes dos resultados**
- A base (Kaggle, CC0, versão 1) reúne **projetos genéricos de infraestrutura — não é uma amostra específica de data centers**. Os resultados são comparações relativas a essa distribuição.
- Percentil empírico = fração dos projetos históricos com valor **menor ou igual** ao do cenário. **Não é probabilidade de viabilidade nem de fracasso.**
- O índice (média simples dos três percentis), os pesos iguais e as faixas Favorável (< 33), Atenção (33 a < 67) e Crítica (≥ 67) são **heurísticas do MVP original**, não validadas por especialistas. Nenhum modelo de aprendizado de máquina é usado.
- Os cenários das seções P2–P4 são **hipotéticos e ilustrativos**.
- Cada seção separa **resultado observado** (calculado aqui) de **interpretação** e de **premissas**.

## 0. Preparação

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys, urllib.request
from pathlib import Path

BASE_URL = ('https://raw.githubusercontent.com/ricardopimentelandre/'
            'MVP-Viabilidade-Nova-Unidade-Dados/psp8-etapa-2-pipeline/scripts/')
Path('/content/scripts').mkdir(exist_ok=True)
for nome in ('pipeline_etl_drive.py', 'analise_ssd.py'):
    urllib.request.urlretrieve(BASE_URL + nome, f'/content/scripts/{nome}')
sys.path.insert(0, '/content')
print('Código obtido:', sorted(p.name for p in Path('/content/scripts').glob('*.py')))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scripts.analise_ssd import *
from scripts.pipeline_etl_drive import SHA256_CSV_V1

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
AZUL, LARANJA, VERDE, VERMELHO, CINZA = '#0072B2', '#E69F00', '#009E73', '#D55E00', '#555555'
COR_DIM = {'custo': AZUL, 'risco': VERMELHO, 'prazo': VERDE}
NOME_DIM = {'custo': 'Custo', 'risco': 'Risco', 'prazo': 'Prazo'}
pd.options.display.float_format = '{:,.2f}'.format

# --- Escolha da execução (camada prata verificada no Drive) ---
RAIZ_EXECUCOES = Path('/content/drive/MyDrive/UnB/PSP8/MVP_Viabilidade_Nova_Unidade/execucoes')
PASTA_EXECUCAO = None   # ou Path('.../execucoes/<id>') para fixar uma execução específica
if PASTA_EXECUCAO is None:
    candidatas = sorted(p for p in RAIZ_EXECUCOES.iterdir()
                        if (p / 'evidencias' / 'verificacao_persistencia.json').is_file())
    if not candidatas:
        raise RuntimeError('Nenhuma execução com verificacao_persistencia.json. Rode antes o notebook do pipeline.')
    PASTA_EXECUCAO = candidatas[-1]

df, manifesto = carregar_prata(PASTA_EXECUCAO)   # confere SHA-256 e contagens com o manifesto; não grava nada
ANALISES = PASTA_EXECUCAO / 'analises'
ANALISES.mkdir(exist_ok=True)

DADOS_CONFEREM_V1 = manifesto['sha256_prata'] == SHA256_CSV_V1
print('Execução usada      :', PASTA_EXECUCAO.name)
print('Registros / colunas :', len(df), '/', df.shape[1])
print('SHA-256 da prata    :', manifesto['sha256_prata'])
print('Idêntica à v1 do Kaggle (hash documentado):', DADOS_CONFEREM_V1)
if not DADOS_CONFEREM_V1:
    print('ATENÇÃO: os dados diferem da v1; os números citados nos textos de discussão podem não valer.')

def salvar(fig, nome):
    fig.savefig(ANALISES / nome, dpi=150, bbox_inches='tight')
    plt.show()

## P1 — Comportamento histórico de custos, riscos e prazos
**Método:** estatísticas descritivas (mínimo, quartis, máximo, média, desvio-padrão, coeficiente de variação, assimetria), histogramas, correlações e médias por `Feasibility_Label`. Todas as 3.245 linhas da prata são usadas; nenhuma foi removida.

In [ ]:
est = estatisticas_descritivas(df)
est.index = ['Custo (US$)', 'Risco (escore)', 'Prazo (dias)']
display(est.T)
est.to_csv(ANALISES / 'p1_estatisticas_descritivas.csv')

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(14, 4))
for ax, dim in zip(eixos, DIMENSOES):
    s = df[DIMENSOES[dim]['coluna']].astype(float)
    escala = 1e6 if dim == 'custo' else 1
    ax.hist(s / escala, bins=30, color=COR_DIM[dim], alpha=0.8, edgecolor='white')
    ax.axvline(s.median() / escala, color='black', ls='--', lw=1.5, label=f'mediana = {s.median()/escala:,.1f}')
    ax.axvline(s.mean() / escala, color=LARANJA, ls=':', lw=2, label=f'média = {s.mean()/escala:,.1f}')
    ax.set_title(f'{NOME_DIM[dim]}: distribuição histórica')
    ax.set_xlabel('US$ milhões' if dim == 'custo' else ('escore (0–100)' if dim == 'risco' else 'dias'))
    ax.set_ylabel('nº de projetos')
    ax.legend(fontsize=9)
fig.suptitle('P1 — Distribuição de custo, risco e prazo (n = 3.245 projetos genéricos de infraestrutura)', y=1.03)
salvar(fig, 'p1_histogramas.png')

In [ ]:
print('Correlação de Pearson entre as dimensões:')
display(correlacoes(df).round(3))
print('Médias por Feasibility_Label (rótulo da fonte, não é verdade de campo):')
display(resumo_por_rotulo(df))
correlacoes(df).to_csv(ANALISES / 'p1_correlacoes.csv')
resumo_por_rotulo(df).to_csv(ANALISES / 'p1_medias_por_rotulo.csv')

### Discussão P1
**Observado** (calculado acima sobre a prata, SHA-256 `a93b9f10…`):
- **Custo:** de US$ 1,00 mi a US$ 7,00 mi; mediana US$ 3,99 mi, média US$ 4,01 mi, desvio-padrão US$ 1,71 mi (CV 42,6 %).
- **Risco:** de 30 a 99; mediana 64, média 64,6, desvio-padrão 20,2 (CV 31,3 %).
- **Prazo:** de 120 a 599 dias; mediana 356, média 357,5, desvio-padrão 141,0 (CV 39,5 %).
- Assimetria ≈ 0 nas três dimensões, **nenhum outlier** pelo critério 1,5×IQR e mediana ≈ média: as distribuições são **simétricas e aproximadamente planas** (os quartis dividem a faixa em partes quase iguais), sem cauda longa nem concentração.
- As correlações lineares entre custo, risco e prazo são ≈ 0 (|r| < 0,02): na base, **não há correlação linear relevante entre as três dimensões**.
- As médias das três dimensões são praticamente iguais entre os rótulos Feasible, Not Feasible e Borderline (por exemplo, risco médio 64,0 / 64,8 / 66,6).

**Interpretação (hipótese, não comprovada pela base):** o formato plano, as faixas "redondas" e a ausência de correlação linear entre as dimensões são típicos de dados gerados artificialmente; a fonte não documenta a origem. Isso limita a leitura dos resultados como retrato de projetos reais.

**Resposta à P1:** na base, custo, risco e prazo se espalham de forma ampla e simétrica, sem projetos atípicos e sem correlação linear relevante entre si. Um valor "alto" ou "baixo" tem, portanto, um significado puramente posicional dentro dessa faixa.

## P2 — Posição de um cenário frente ao histórico (percentis)
**Cenário hipotético (o mesmo do MVP original):** custo US$ 5.500.000, risco 80/100, prazo 480 dias. Os valores podem ser alterados na célula abaixo.

**Método:** percentil empírico de cada dimensão = `(histórico ≤ valor).mean() × 100`, calculado sobre a prata relida do Drive.

In [ ]:
CUSTO, RISCO, PRAZO = 5_500_000, 80, 480
BASE = {'custo': CUSTO, 'risco': RISCO, 'prazo': PRAZO}
cenario = avaliar_cenario(df, **BASE)

tab = pd.DataFrame({
    'valor do cenário': [CUSTO, RISCO, PRAZO],
    'mediana histórica': [df[DIMENSOES[d]['coluna']].median() for d in DIMENSOES],
    'percentil (%)': [cenario['percentis'][d] for d in DIMENSOES],
    'projetos históricos com valor ≤ ao do cenário': [int((df[DIMENSOES[d]['coluna']] <= BASE[d]).sum()) for d in DIMENSOES],
}, index=['Custo (US$)', 'Risco (escore)', 'Prazo (dias)'])
display(tab)
tab.to_csv(ANALISES / 'p2_percentis_do_cenario.csv')

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(14, 4))
for ax, dim in zip(eixos, DIMENSOES):
    s = df[DIMENSOES[dim]['coluna']].astype(float).sort_values().reset_index(drop=True)
    escala = 1e6 if dim == 'custo' else 1
    y = (s.rank(method='max') / len(s) * 100)
    ax.plot(s / escala, y, color=COR_DIM[dim], lw=2.5)
    v, p = BASE[dim] / escala, cenario['percentis'][dim]
    ax.plot([v, v], [0, p], color='black', ls='--', lw=1.3)
    ax.plot([s.min() / escala, v], [p, p], color='black', ls='--', lw=1.3)
    ax.scatter([v], [p], color='black', zorder=5, s=50)
    ax.annotate(f'cenário: {v:,.1f}\npercentil {p:.1f}', (v, p), xytext=(-110, -50) if p > 60 else (10, 10),
                textcoords='offset points', fontsize=10, bbox=dict(boxstyle='round', fc='white', ec='gray'))
    ax.set_title(f'{NOME_DIM[dim]}: onde o cenário se posiciona')
    ax.set_xlabel('US$ milhões' if dim == 'custo' else ('escore (0–100)' if dim == 'risco' else 'dias'))
    ax.set_ylabel('percentil acumulado (%)')
    ax.set_ylim(0, 102)
fig.suptitle('P2 — Distribuição acumulada histórica e posição do cenário (percentil ≠ probabilidade)', y=1.03)
salvar(fig, 'p2_percentis_do_cenario.png')

### Discussão P2
**Observado:** o cenário de US$ 5,5 mi, risco 80 e 480 dias fica nos percentis **75,7** (custo), **72,3** (risco) e **75,1** (prazo): cerca de três em cada quatro projetos históricos têm valor menor ou igual ao do cenário em cada dimensão. As três dimensões ficam bem acima da mediana (percentil 50).

**Interpretação:** o cenário é "alto" de forma equilibrada nas três dimensões; nenhuma delas se destaca como a pior. Como não há correlação linear relevante entre as dimensões na base (P1), ela não mostra que projetos caros sejam também longos ou arriscados.

**Cuidados:** o percentil descreve **posição relativa** em uma amostra de projetos genéricos de infraestrutura. **Não** informa a probabilidade de o cenário ser viável ou inviável, nem o custo real de uma unidade de dados. Um percentil alto de custo ou de prazo não significa, por si só, inviabilidade.

**Resposta à P2:** o cenário demonstrado se posiciona bem acima da mediana nas três dimensões, próximo ao terceiro quartil (percentis 72–76).

## P3 — Índice geral de exposição e recomendações
**Método (heurística do MVP original, inalterada):** índice = média simples (pesos iguais) dos percentis de custo, risco e prazo. Faixas: **Favorável** < 33 · **Atenção** 33 a < 67 · **Crítica** ≥ 67. Recomendação específica por dimensão quando o percentil dela é ≥ 67. *Pesos e cortes são premissas de projeto, não resultados.*

In [ ]:
print(f"Índice geral de exposição: {cenario['indice']:.1f}  (média de {', '.join(f'{p:.1f}' for p in cenario['percentis'].values())})")
print(f"Faixa: {cenario['faixa']}")
print('\nContribuição de cada dimensão ao índice (%):', {NOME_DIM[k]: round(v, 1) for k, v in cenario['contribuicao_%'].items()})
print('\nRecomendações:')
for r in cenario['recomendacoes']:
    print(' -', r)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.axvspan(0, CORTE_FAVORAVEL, color=VERDE, alpha=0.15, label='Favorável (< 33)')
ax.axvspan(CORTE_FAVORAVEL, CORTE_CRITICA, color=LARANJA, alpha=0.15, label='Atenção (33 a < 67)')
ax.axvspan(CORTE_CRITICA, 100, color=VERMELHO, alpha=0.15, label='Crítica (≥ 67)')
nomes = ['Custo', 'Risco', 'Prazo', 'Índice geral']
valores = [*cenario['percentis'].values(), cenario['indice']]
cores = [COR_DIM['custo'], COR_DIM['risco'], COR_DIM['prazo'], 'black']
barras = ax.barh(nomes[::-1], valores[::-1], color=cores[::-1], height=0.55)
for b, v in zip(barras, valores[::-1]):
    ax.text(v + 1, b.get_y() + b.get_height() / 2, f'{v:.1f}', va='center', fontweight='bold')
ax.set_xlim(0, 100); ax.set_xlabel('percentil / índice (0–100)')
ax.set_title(f"P3 — Índice de exposição do cenário: {cenario['indice']:.1f} → {cenario['faixa']}")
ax.legend(loc='lower right', fontsize=9)
salvar(fig, 'p3_indice_do_cenario.png')

In [ ]:
hist = indice_historico(df)
dist = hist['faixa'].value_counts(normalize=True).mul(100).reindex(['FAVORÁVEL', 'ATENÇÃO', 'CRÍTICA'])
print('Como a heurística classifica os próprios projetos históricos (%):')
display(dist.to_frame('% dos projetos'))
print('Índice médio por Feasibility_Label (rótulo da fonte):')
display(hist.groupby('Feasibility_Label')['indice'].agg(['count', 'mean', 'median']))
print('Distribuição das faixas por rótulo (% da linha):')
display(pd.crosstab(hist['Feasibility_Label'], hist['faixa'], normalize='index').mul(100)
        [['FAVORÁVEL', 'ATENÇÃO', 'CRÍTICA']])
hist.to_csv(ANALISES / 'p3_indice_historico.csv', index=False)

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.hist(hist['indice'], bins=40, color=CINZA, alpha=0.8, edgecolor='white')
ax.axvline(CORTE_FAVORAVEL, color=VERDE, lw=2); ax.axvline(CORTE_CRITICA, color=VERMELHO, lw=2)
ax.axvline(cenario['indice'], color='black', ls='--', lw=2, label=f"cenário = {cenario['indice']:.1f}")
ax.set_xlabel('índice de exposição'); ax.set_ylabel('nº de projetos históricos')
ax.set_title('P3 — Índice dos projetos históricos e posição do cenário'); ax.legend()
salvar(fig, 'p3_indice_historico.png')

### Discussão P3
**Observado:** o cenário tem índice **74,4** e cai na faixa **Crítica**. As três dimensões contribuem de forma quase igual (custo 33,9 %, risco 32,4 %, prazo 33,7 %), e as três atingem o limiar de recomendação (percentil ≥ 67): revisar orçamento e criar reserva de contingência, elaborar plano de mitigação de riscos, dividir a implantação em etapas e **não iniciar a implantação sem revisão do planejamento**.

Aplicando o mesmo índice a cada projeto histórico: **15,3 %** ficam em Favorável, **67,9 %** em Atenção e **16,7 %** em Crítica; o índice médio é 50,3. Ou seja, o índice do cenário é maior ou igual ao de 92,3 % dos projetos históricos (e maior que o 3º quartil, 61,7); 7,7 % dos projetos têm índice maior que 74,4.

**Resultado negativo relevante:** **não há associação descritiva relevante** entre o índice e o `Feasibility_Label` da fonte. O índice médio é 51,2 (Borderline), 50,2 (Feasible) e 49,9 (Not Feasible), e a distribuição nas faixas é quase idêntica entre os rótulos (Crítica: 16,9 % / 17,3 % / 15,3 %). Isso **não valida nem invalida** o índice (o rótulo da fonte não é verdade de campo e sua origem não é documentada), mas mostra que, **nesta base, o índice não reproduz a classificação de viabilidade da fonte** e não deve ser lido como previsão dela.

**Interpretação e limites:** o índice é uma **triagem relativa**: indica que o cenário é pouco comum frente ao histórico e quais dimensões revisar antes de detalhar o planejamento. Pesos iguais e cortes 33/67 são premissas; com outros pesos ou cortes a faixa pode mudar. A decisão final permanece humana.

**Resposta à P3:** exposição geral **74,4 (Crítica)**, com revisão recomendada de custo, risco e prazo, e de todo o planejamento, tratando-se de uma triagem heurística e não de uma previsão de viabilidade.

## P4 — Sensibilidade a mudanças de custo, risco e prazo
**Método (reproduzível):** a partir do cenário-base, (a) comparam-se cenários hipotéticos nomeados; (b) varia-se **uma dimensão por vez** (as outras ficam no valor-base) em ±10 %, ±20 % e ±30 % do valor-base (risco limitado a 100); (c) repete-se a variação em desvios-padrão históricos, que torna as dimensões comparáveis; (d) varre-se cada dimensão por toda a faixa observada; (e) calcula-se o **ponto de virada**: o maior valor da dimensão que mantém o índice abaixo de 67. Os cenários e as variações são **escolhas ilustrativas nossas**.

In [ ]:
CENARIOS = {
    'Base':                 (5_500_000, 80, 480),
    'A: custo −20 %':       (4_400_000, 80, 480),
    'B: risco 80 → 60':     (5_500_000, 60, 480),
    'C: prazo −20 %':       (5_500_000, 80, 384),
    'D: A + B + C':         (4_400_000, 60, 384),
    'E: piora +20 %':       (6_600_000, 96, 576),
    'F: perfil mediano':    (df['Estimated_Cost_USD'].median(), df['Risk_Assessment_Score'].median(), df['Time_Estimate_Days'].median()),
}
comp = comparar_cenarios(df, CENARIOS)
display(comp)
comp.to_csv(ANALISES / 'p4_comparacao_de_cenarios.csv')

fig, ax = plt.subplots(figsize=(9.5, 4.5))
ax.axhspan(0, CORTE_FAVORAVEL, color=VERDE, alpha=0.15); ax.axhspan(CORTE_FAVORAVEL, CORTE_CRITICA, color=LARANJA, alpha=0.15)
ax.axhspan(CORTE_CRITICA, 100, color=VERMELHO, alpha=0.15)
cores = {'FAVORÁVEL': VERDE, 'ATENÇÃO': LARANJA, 'CRÍTICA': VERMELHO}
barras = ax.bar(range(len(comp)), comp['indice'], color=[cores[f] for f in comp['faixa']], edgecolor='black', width=0.6)
for i, (v, f) in enumerate(zip(comp['indice'], comp['faixa'])):
    ax.text(i, v + 1.2, f'{v:.1f}', ha='center', fontweight='bold')
ax.set_xticks(range(len(comp))); ax.set_xticklabels(comp.index, rotation=25, ha='right')
ax.set_ylim(0, 100); ax.set_ylabel('índice de exposição')
ax.set_title('P4 — Índice de exposição por cenário (fundo: faixas heurísticas 33 e 67)')
salvar(fig, 'p4_comparacao_de_cenarios.png')

In [ ]:
VARIACOES = [-30, -20, -10, 0, 10, 20, 30]
sens = sensibilidade_uma_dimensao(df, BASE, VARIACOES)
piv = sens.pivot(index='variacao_%', columns='dimensao', values='indice')[['custo', 'risco', 'prazo']]
print('Índice quando UMA dimensão varia (variação em % do valor-base):')
display(piv)
faixas = sens.pivot(index='variacao_%', columns='dimensao', values='faixa')[['custo', 'risco', 'prazo']]
display(faixas)
sens.to_csv(ANALISES / 'p4_sensibilidade_percentual.csv', index=False)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.axhspan(CORTE_CRITICA, 100, color=VERMELHO, alpha=0.12); ax.axhspan(CORTE_FAVORAVEL, CORTE_CRITICA, color=LARANJA, alpha=0.12)
for dim in DIMENSOES:
    ax.plot(piv.index, piv[dim], marker='o', lw=2.2, color=COR_DIM[dim], label=NOME_DIM[dim])
ax.axhline(CORTE_CRITICA, color=VERMELHO, ls='--', lw=1.2)
ax.set_xlabel('variação da dimensão (% do valor-base)'); ax.set_ylabel('índice de exposição'); ax.set_ylim(55, 90)
ax.set_title('P4 — Sensibilidade: uma dimensão varia, as outras ficam no cenário-base'); ax.legend()
salvar(fig, 'p4_sensibilidade_percentual.png')

In [ ]:
desv = sensibilidade_em_desvios(df, BASE, [-1, -0.5, 0, 0.5, 1])
pivd = desv.pivot(index='desvios', columns='dimensao', values='indice')[['custo', 'risco', 'prazo']]
print('Índice quando UMA dimensão varia em desvios-padrão históricos (comparável entre dimensões):')
display(pivd)
desv.to_csv(ANALISES / 'p4_sensibilidade_em_desvios.csv', index=False)

fig, eixos = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, dim in zip(eixos, DIMENSOES):
    c = curva_indice(df, BASE, dim)
    escala = 1e6 if dim == 'custo' else 1
    ax.plot(c['valor'] / escala, c['indice'], color=COR_DIM[dim], lw=2.5)
    ax.axhline(CORTE_CRITICA, color=VERMELHO, ls='--', lw=1.2); ax.axhline(CORTE_FAVORAVEL, color=VERDE, ls='--', lw=1.2)
    ax.axvline(BASE[dim] / escala, color='black', ls=':', lw=1.5, label='valor-base')
    ax.set_xlabel('US$ milhões' if dim == 'custo' else ('escore (0–100)' if dim == 'risco' else 'dias'))
    ax.set_title(f'{NOME_DIM[dim]} varia na faixa observada'); ax.legend(fontsize=9)
eixos[0].set_ylabel('índice de exposição'); eixos[0].set_ylim(30, 100)
fig.suptitle('P4 — Índice × valor da dimensão (outras fixas no cenário-base; tracejado: cortes 33 e 67)', y=1.03)
salvar(fig, 'p4_curvas_por_dimensao.png')

In [ ]:
pv = pd.DataFrame([ponto_de_virada(df, BASE, d, CORTE_CRITICA) for d in DIMENSOES]).set_index('dimensao')
pv['valor-base'] = [BASE[d] for d in pv.index]
display(pv[['valor-base', 'valor', 'variacao_%', 'alcancavel']].rename(
    columns={'valor': 'maior valor com índice < 67', 'variacao_%': 'variação necessária (%)', 'alcancavel': 'alcançável?'}))
pv33 = pd.DataFrame([ponto_de_virada(df, BASE, d, CORTE_FAVORAVEL) for d in DIMENSOES]).set_index('dimensao')
print('Alcançaria "Favorável" (< 33) reduzindo apenas UMA dimensão, até o mínimo observado?')
display(pv33[['alcancavel']])
pv.to_csv(ANALISES / 'p4_pontos_de_virada.csv')

print('Recomendações por cenário:')
for nome, (c, r, p) in CENARIOS.items():
    a = avaliar_cenario(df, c, r, p)
    print(f"\n{nome} → índice {a['indice']:.1f} ({a['faixa']})")
    for rec in a['recomendacoes'][:3]:
        print('   -', rec)
    print('   =>', a['recomendacoes'][3])

### Discussão P4
**Observado** (cenário-base: índice 74,4, Crítica):
- **Reduzir uma dimensão isolada em 20 %** baixa o índice para 68,2 (custo), 67,0 (risco: 80 → 64) e 67,9 (prazo): em todos os casos o índice cai entre 6,2 e 7,4 pontos, mas **só o risco cruza a fronteira para Atenção**, e por muito pouco (índice 66,985, exibido como 66,99; a faixa é decidida com o valor **sem arredondar**; custo e prazo ficam em 68,2 e 67,9, ainda Crítica).
- **Pontos de virada** (maior valor que mantém o índice abaixo de 67, com as outras dimensões fixas): custo ≈ US$ 4,22 mi (−23,2 %), risco 64 (−20,0 %), prazo 370 dias (−22,9 %). Nenhuma redução isolada, mesmo até o mínimo observado, leva o índice abaixo de 33 (Favorável): o menor índice alcançado é ≈ 49–51.
- **Combinar** redução de custo (−20 %), de risco (80 → 60) e de prazo (−20 %) leva a 52,7 (Atenção, sem nenhum alerta por dimensão). O cenário de **piora de 20 %** sobe a 94,6 (Crítica).
- **Em desvios-padrão** (comparação sem efeito de escala), subir ou descer 1 desvio mexe o índice quase igual nas três dimensões (de 64,6–64,9 a 82,5–83,6): **nenhuma dimensão domina**. Varrendo qualquer dimensão do mínimo ao máximo observado, o índice vai de ≈ 49–51 a ≈ 82–84, e a curva é **praticamente linear** (r ≥ 0,9998), consequência de a distribuição ser plana. O único achatamento visível na tabela percentual (+30 %) vem de o percentil não passar de 100 acima do valor máximo observado.
- O cenário-mediana (F) tem índice 50,1, o que confere com a construção do índice (percentil 50 nas três dimensões).

**Interpretação:** com pesos iguais e dimensões sem correlação linear relevante e de formato plano na base, o índice reage de modo **aproximadamente simétrico e proporcional** a cada dimensão: a sensibilidade em % do valor-base difere um pouco só porque o risco tem faixa mais estreita (30–99). Logo, o índice **não indica por si só qual alavanca priorizar**: essa escolha depende de custo e viabilidade de agir em cada dimensão, informação que a base não traz. O que a análise oferece é a **ordem de grandeza** — mudanças de 20 % em uma dimensão isolada deslocam o índice em 6,2 a 7,4 pontos, o que deixa o cenário-base (74,4) perto da fronteira de 67.

**Limites:** a análise é determinística sobre a base (sem incerteza estatística), os cenários e as variações são escolhas ilustrativas, os pesos iguais e os cortes são premissas, e a base é de projetos genéricos (não de data centers). Pesos diferentes mudariam as conclusões de sensibilidade.

**Resposta à P4:** reduções isoladas de ~20 % em qualquer das três dimensões aproximam o cenário da fronteira Crítica/Atenção sem torná-lo Favorável; apenas combinar reduções nas três leva a Atenção com folga. As três dimensões têm peso de influência semelhante no índice.

## Discussão geral e limitações
- **Resposta ao problema:** em relação ao histórico da base, o cenário de US$ 5,5 mi, risco 80 e 480 dias é **alto nas três dimensões** frente ao histórico (percentis 72–76; índice 74,4, Crítica). A triagem sugere revisar custo, risco e prazo antes do planejamento detalhado, e mostra que reduções isoladas de ~20 % não bastam para torná-lo favorável. A **decisão final permanece humana**.
- **O que o MVP não demonstra:** probabilidade de viabilidade, eficácia em data centers reais ou equivalência com o `Feasibility_Label` da fonte (nesta base, não há associação descritiva relevante entre o índice e o rótulo).
- **Limitações dos dados:** projetos genéricos de infraestrutura (não data centers); procedência não documentada; indícios de dado sintético (distribuições planas e ausência de correlação linear relevante entre as dimensões) — hipótese, não fato; sem dimensão temporal.
- **Limitações do método:** percentil é posição relativa; pesos iguais e cortes 33/67 são heurísticas; o risco é assumido numa escala 0–100 em que maior valor significa mais risco (não documentado pela fonte).

In [ ]:
print('Arquivos de análise salvos no Drive em:', ANALISES)
for p in sorted(ANALISES.iterdir()):
    print(' -', p.name, f'({p.stat().st_size:,} bytes)')